# 02 - Pre-training: watching gibberish become English  *(~3 minutes)*

> **Presenter script.** "Right. We have clean data and a validation set locked
> in a drawer. Now we build a model that knows literally nothing, and we watch
> it learn to write. This is the bit people think is magic. It is one idea
> repeated a lot of times: **guess the next character, then adjust.**"

> **pre-training** - teaching a model the shape of language itself, by making it
> predict the next token over and over on a large pile of text. No questions, no
> answers, no instructions. Just "what comes next?"

In [ ]:
# --- boilerplate: make `import minigpt` work no matter where Jupyter started ---
import pathlib
import sys

ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "minigpt").is_dir())
sys.path.insert(0, str(ROOT))

import torch

torch.set_num_threads(4)  # plenty for a model this small; more threads is not faster
print("repo root:", ROOT)

In [ ]:
from minigpt import data
from minigpt import train as T
from minigpt.model import DEFAULT_BATCH_SIZE, MiniGPT, default_config
from minigpt.plots import plot_curves, use_stream_style

import math
import matplotlib.pyplot as plt

use_stream_style()

## Presenter switch

`USE_PREBAKED = True` skips the live training and loads the checkpoint that
`scripts/pretrain_all.py` produced earlier. Flip it if the room's CPU turns out
to be slower than you hoped.

In [ ]:
USE_PREBAKED = False   # <-- the safety net
STEPS = 1200           # about 2-3 minutes on a modest CPU

## 1. Load the data we prepared in notebook 01

In [ ]:
tokenizer = data.load_tokenizer()
train_text = data.load_split("base", "train")
val_text = data.load_split("base", "val")

train_data = T.encode_to_tensor(train_text, tokenizer)
val_data = T.encode_to_tensor(val_text, tokenizer)

print(f"vocabulary : {tokenizer.vocab_size} tokens")
print(f"train      : {len(train_data):,} tokens")
print(f"validation : {len(val_data):,} tokens  (the model will never train on these)")

## 2. Build the model

Here is every knob, in plain English. These are the **hyperparameters** -
settings *you* choose before training, as opposed to the parameters the model
learns for itself.

| knob | what it means | ours |
|---|---|---|
| `n_layer` | how many rounds of "look back, then think" | 3 |
| `n_head` | how many things it can pay attention to at once | 4 |
| `n_embd` | how wide the model is - numbers used per token | 64 |
| `block_size` | **context window**: how far back it can see | 96 characters |
| `dropout` | randomly ignore 10% of internal values while training, to discourage memorising | 0.1 |

Those choices give us a model of about **165,000 parameters**. A parameter is
one adjustable number. GPT-3 has 175 *billion*. Ours would fit in a large
spreadsheet.

In [ ]:
T.set_seed(1337)   # same seed -> same run, every time

config = default_config(tokenizer.vocab_size)
model = MiniGPT(config)

print(config)
print(f"\nparameters: {model.num_params():,}")
print(f"that is about 1/{175_000_000_000 // model.num_params():,}th the size of GPT-3")

## 3. What does an untrained model sound like?

Every parameter is currently a small random number. Let's ask it to write.

In [ ]:
sample_step_0 = T.show_sample(model, tokenizer, "\n", 220, label="STEP 0 - before any training")

That is not a bug. That is what "knows nothing" looks like: the model is picking
characters more or less uniformly at random.

## 4. Loss - the number that tells us how it is going

> **loss** - "how surprised was the model by the character that actually came
> next?" Lower is better.

We can compute exactly what a *completely random* model should score. With
`V` equally likely characters, the loss is `ln(V)`. Anything below that line
means the model has genuinely learned something.

In [ ]:
random_baseline = math.log(tokenizer.vocab_size)
print(f"vocabulary size          : {tokenizer.vocab_size}")
print(f"loss if guessing randomly: ln({tokenizer.vocab_size}) = {random_baseline:.2f}")

start_loss = T.evaluate(model, val_data, config.block_size, 32, 16, seed=0)
print(f"our untrained model      : {start_loss:.2f}   <- right on the line, as expected")

## 5. Train it

A few more words we are about to use:

* **step** (or *iteration*) - look at one batch, make one small adjustment.
* **batch size** - how many chunks of text we look at per step (12 for us).
* **learning rate** - how big each adjustment is. Ours starts at `0.003`.
* **epoch** - one full pass over the training set. We count steps instead,
  because our corpus is small and we go round it several times.

Every 100 steps we pause and take the **pop quiz** on the validation set, and at
step 0, half way and at the end we print a writing sample.

> **Presenter note.** This cell takes about 2-3 minutes. It is the one place in
> the stream where you should talk over a progress log. Watch the two numbers
> fall *together* - that is what healthy learning looks like.

In [ ]:
if USE_PREBAKED:
    model, tokenizer, _ = T.load_checkpoint("base")
    history = T.load_history("healthy")
    print("loaded the pre-baked run (skipped ~2 minutes of training)")
    for point in history.samples:
        print("\n" + "=" * 72)
        print(f"SAMPLE AT STEP {point['step']}")
        print("=" * 72)
        print(point["text"])
else:
    history = T.History(name="healthy")

    def snapshot(step, train_loss, val_loss):
        """Print a writing sample at the start, the middle and the end."""
        if step in (0, STEPS // 2, STEPS):
            text = T.generate(model, tokenizer, "\n", 220, seed=1337)
            history.add_sample(step, text)
            print("\n" + "=" * 72)
            print(f"SAMPLE AT STEP {step}")
            print("=" * 72)
            print(text)
            print("=" * 72 + "\n")

    T.train_model(
        model, train_data, val_data,
        steps=STEPS,
        batch_size=DEFAULT_BATCH_SIZE,
        learning_rate=3e-3,
        eval_every=100,
        eval_iters=16,
        eval_batch_size=32,
        seed=1337,
        name="healthy",
        history=history,
        on_eval=snapshot,
    )

### Read the three samples back

* **step 0** - random characters. No words.
* **half way** - word-shaped things. Correct letter frequencies, plausible
  spelling, no meaning.
* **the end** - actual words, actual sentences, in the style of the corpus.

Nobody told the model what a word is. It worked that out from "guess the next
character" alone.

## 6. The chart

Two lines. The blue one is how well it does on the homework; the red one is the
pop quiz it has never seen. When both fall together, the model is **learning the
language**, not memorising the text.

In [ ]:
plot_curves(history, title="Pre-training run: both curves fall together", random_baseline=random_baseline)
plt.show()

print(f"start        : {history.val_loss[0]:.3f}")
print(f"end          : {history.val_loss[-1]:.3f}")
print(f"train-val gap: {history.final_gap:+.3f}   <- small gap = healthy")

## 7. Play with the temperature

> **temperature** - how adventurous the model is when it picks the next
> character. Below 1 it plays safe and repeats itself; above 1 it takes risks
> and starts inventing words.

In [ ]:
for temperature in (0.4, 0.8, 1.2):
    T.show_sample(model, tokenizer, "one day ", 180, label=f"temperature = {temperature}",
                  temperature=temperature, seed=7)

## 8. Save the checkpoint

> **checkpoint** - a saved copy of every parameter, plus enough information to
> rebuild the model around them. This file is "the model" - everything from here
> on starts by loading it.

In [ ]:
if not USE_PREBAKED:
    path = T.save_checkpoint(model, tokenizer, "base", extra={"history": "healthy"})
    T.save_history(history)
    print(f"saved {path}  ({path.stat().st_size / 1024:.0f} KB)")
else:
    print("using the pre-baked checkpoints/base.pt - nothing to save")

## Recap

* Pre-training = "predict the next token", repeated a lot.
* **Loss** started at the random-guessing line and fell by ~90%.
* **Validation loss** fell with it, which is the part that matters.
* The model learned words, spacing and sentence shape with nobody defining any
  of them.

**Next:** `03_diagnosis.ipynb` - the most useful notebook in this repo. How to
look at a chart and know what to do next.